# Experiment 3
## Implementation of Convolutional Neural Networks (CNNs) for Image Classification

**CS3807 - Deep Learning Laboratory**

### 1. Import Libraries

In [ ]:
# ==========================================
# Task 1 : Import Required Libraries
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

import torchvision
import torchvision.transforms as transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import random
import time


### 2. Device Configuration

In [ ]:
# ==========================================
# Task 1.1 : Device Configuration and Reproducibility
# ==========================================

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("DEVICE CONFIGURATION")
print("=" * 60)
print("Using device :", device)
if torch.cuda.is_available():
    print("GPU Name     :", torch.cuda.get_device_name(0))


### 3. Load CIFAR-10 Dataset

In [ ]:
# ==========================================
# Task 1.2 : Load CIFAR-10 Dataset
# ==========================================

transform_basic = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_basic
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_basic
)

class_names = train_dataset.classes

print("=" * 60)
print("CIFAR-10 DATASET LOADED SUCCESSFULLY")
print("=" * 60)
print("Training Samples :", len(train_dataset))
print("Testing Samples  :", len(test_dataset))
print("Image Shape      :", train_dataset[0][0].shape)
print("Number of Classes:", len(class_names))
print("Class Names      :", class_names)


### 4. Display Ten Sample Images

In [ ]:
# ==========================================
# Task 1.3 : Display Ten Sample Images
# ==========================================

plt.figure(figsize=(14, 6))

for index in range(10):
    image, label = train_dataset[index]
    image = image.permute(1, 2, 0).numpy()

    plt.subplot(2, 5, index + 1)
    plt.imshow(image)
    plt.title(class_names[label], fontsize=10)
    plt.axis("off")

plt.suptitle("Sample Images from CIFAR-10 Dataset", fontsize=13)
plt.tight_layout()
plt.savefig("plot1.png", dpi=300)
plt.show()


### 5. Dataset Information

In [ ]:
# ==========================================
# Task 1.4 : Dataset Information
# ==========================================

sample_image, _ = train_dataset[0]

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print("Dataset Name          :", "CIFAR-10")
print("Image Dimensions      :", tuple(sample_image.shape))
print("Number of Classes     :", len(class_names))
print("Number of Train Images:", len(train_dataset))
print("Number of Test Images :", len(test_dataset))
print("Pixel Value Range     : [", sample_image.min().item(), ",", sample_image.max().item(), "]")


### 6. Class Distribution

In [ ]:
# ==========================================
# Task 1.5 : Class Distribution
# ==========================================

train_labels_array = np.array(train_dataset.targets)
class_count = np.bincount(train_labels_array)

plt.figure(figsize=(11, 5))
bars = plt.bar(class_names, class_count, color="steelblue")
plt.title("Class Distribution of CIFAR-10 Training Data")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=30)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 20, str(height),
              ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("plot2.png", dpi=300)
plt.show()

print("Class-wise image counts :")
for name, count in zip(class_names, class_count):
    print(f"{name:12s} : {count}")


### 7. Convolution Demonstration (Kernel Size Comparison)

In [ ]:
# ==========================================
# Task 2 : Convolution Demonstration - Kernel Size Comparison
# ==========================================

sample_input, _ = train_dataset[0]
sample_input = sample_input.unsqueeze(0)   # shape : (1, 3, 32, 32)

kernel_sizes = [3, 5, 7]
conv_results = []

for k in kernel_sizes:
    conv_layer = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=k, stride=1, padding=0)
    output = conv_layer(sample_input)
    conv_results.append({
        "Kernel Size": f"{k}x{k}",
        "Input Size": "32x32",
        "Output Size": f"{output.shape[2]}x{output.shape[3]}",
        "Output Channels": output.shape[1]
    })

conv_comparison_df = pd.DataFrame(conv_results)

print("=" * 60)
print("CONVOLUTION KERNEL SIZE COMPARISON")
print("=" * 60)
print(conv_comparison_df.to_string(index=False))


### 8. Hyperparameter Study (Stride and Padding)

In [ ]:
# ==========================================
# Task 3 : Hyperparameter Study - Stride and Padding
# ==========================================

def output_size(n, f, p, s):
    return int(np.floor((n - f + 2 * p) / s) + 1)

N = 32
F_size = 3

hyperparam_results = []

for stride in [1, 2]:
    p_valid = 0
    out_valid = output_size(N, F_size, p_valid, stride)
    hyperparam_results.append({
        "Stride": stride,
        "Padding": "Valid",
        "P": p_valid,
        "Output Size": f"{out_valid}x{out_valid}"
    })

    p_same = (F_size - 1) // 2
    out_same = output_size(N, F_size, p_same, stride)
    hyperparam_results.append({
        "Stride": stride,
        "Padding": "Same",
        "P": p_same,
        "Output Size": f"{out_same}x{out_same}"
    })

hyperparam_df = pd.DataFrame(hyperparam_results)

print("=" * 60)
print("STRIDE AND PADDING COMPARISON (Kernel = 3x3, Input = 32x32)")
print("=" * 60)
print(hyperparam_df.to_string(index=False))


### 9. CNN Architecture

In [ ]:
# ==========================================
# Task 6 : CNN Architecture
# ==========================================

class CIFAR10_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CIFAR10_CNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)          # raw logits, softmax applied via loss function / at inference
        return x

model = CIFAR10_CNN().to(device)

print("=" * 60)
print("CNN ARCHITECTURE")
print("=" * 60)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nTotal Parameters     :", total_params)
print("Trainable Parameters :", trainable_params)


### 10. Parameter Calculation

In [ ]:
# ==========================================
# Task 3 (Numerical) : Layer-wise Parameter Calculation
# ==========================================

def conv_params(in_channels, out_channels, kernel_size):
    return (in_channels * kernel_size * kernel_size + 1) * out_channels

def dense_params(in_features, out_features):
    return (in_features + 1) * out_features

conv1_params = conv_params(3, 32, 3)
conv2_params = conv_params(32, 64, 3)
fc1_params = dense_params(64 * 8 * 8, 128)
fc2_params = dense_params(128, 10)

layer_param_df = pd.DataFrame({
    "Layer": ["Conv1", "Conv2", "FC1", "FC2"],
    "Parameters": [conv1_params, conv2_params, fc1_params, fc2_params]
})

print("=" * 60)
print("LAYER-WISE PARAMETER CALCULATION")
print("=" * 60)
print(layer_param_df.to_string(index=False))
print("\nTotal Calculated Parameters :", layer_param_df["Parameters"].sum())
print("Matches model.parameters()  :", layer_param_df["Parameters"].sum() == trainable_params)


### 11. Feature Map Visualization

In [ ]:
# ==========================================
# Task 4 : Feature Map Visualization After First Convolution Layer
# ==========================================

model.eval()
sample_image_fm, sample_label_fm = train_dataset[3]
sample_batch = sample_image_fm.unsqueeze(0).to(device)

with torch.no_grad():
    conv1_output = model.relu1(model.conv1(sample_batch))

conv1_output = conv1_output.squeeze(0).cpu()

plt.figure(figsize=(14, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(conv1_output[i], cmap="viridis")
    plt.title(f"Feature Map {i + 1}", fontsize=9)
    plt.axis("off")

plt.suptitle(f"Feature Maps After Conv1 (Class : {class_names[sample_label_fm]})", fontsize=13)
plt.tight_layout()
plt.savefig("plot3.png", dpi=300)
plt.show()


### 12. Pooling Comparison (Max Pooling vs Average Pooling)

In [ ]:
# ==========================================
# Task 5 : Pooling Comparison
# ==========================================

max_pool_layer = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool_layer = nn.AvgPool2d(kernel_size=2, stride=2)

max_pool_output = max_pool_layer(conv1_output.unsqueeze(0))
avg_pool_output = avg_pool_layer(conv1_output.unsqueeze(0))

pooling_df = pd.DataFrame({
    "Pooling Type": ["Max Pooling", "Average Pooling"],
    "Input Size": [f"{conv1_output.shape[1]}x{conv1_output.shape[2]}"] * 2,
    "Output Size": [f"{max_pool_output.shape[2]}x{max_pool_output.shape[3]}",
                     f"{avg_pool_output.shape[2]}x{avg_pool_output.shape[3]}"],
    "Observation": [
        "Retains the sharpest activations, preserving edges and prominent features",
        "Smooths the activations, retaining background context but losing sharp detail"
    ]
})

print("=" * 60)
print("MAX POOLING vs AVERAGE POOLING")
print("=" * 60)
print(pooling_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(conv1_output[0], cmap="viridis")
axes[0].set_title("Original Feature Map")
axes[0].axis("off")

axes[1].imshow(max_pool_output[0, 0].cpu(), cmap="viridis")
axes[1].set_title("Max Pooled")
axes[1].axis("off")

axes[2].imshow(avg_pool_output[0, 0].cpu(), cmap="viridis")
axes[2].set_title("Average Pooled")
axes[2].axis("off")

plt.tight_layout()
plt.show()


### 13. Training

In [ ]:
# ==========================================
# Task 6 (contd.) : Data Preparation for Training
# ==========================================

batch_size = 32

val_size = 5000
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size],
                                          generator=torch.Generator().manual_seed(seed))

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Training Samples  :", len(train_subset))
print("Validation Samples:", len(val_subset))
print("Testing Samples   :", len(test_dataset))


In [ ]:
# ==========================================
# Task 6 (contd.) : Model Training
# ==========================================

model = CIFAR10_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20

train_loss_history = []
val_loss_history = []
train_acc_history = []
val_acc_history = []

training_start = time.time()

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_acc = correct / total

    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_acc = val_correct / val_total

    train_loss_history.append(epoch_train_loss)
    val_loss_history.append(epoch_val_loss)
    train_acc_history.append(epoch_train_acc)
    val_acc_history.append(epoch_val_acc)

    print(f"Epoch [{epoch + 1:2d}/{num_epochs}]  "
          f"Train Loss: {epoch_train_loss:.4f}  Train Acc: {epoch_train_acc:.4f}  "
          f"Val Loss: {epoch_val_loss:.4f}  Val Acc: {epoch_val_acc:.4f}")

training_end = time.time()
total_training_time = training_end - training_start

print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)
print(f"Total Training Time : {total_training_time:.2f} seconds")


### 14. Training Accuracy Graph

In [ ]:
# ==========================================
# Task 7.1 : Training and Validation Accuracy Plot
# ==========================================

plt.figure(figsize=(8, 5))
plt.plot(train_acc_history, marker="o", label="Training Accuracy")
plt.plot(val_acc_history, marker="s", label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("plot4.png", dpi=300)
plt.show()


### 15. Loss Graph

In [ ]:
# ==========================================
# Task 7.2 : Training and Validation Loss Plot
# ==========================================

plt.figure(figsize=(8, 5))
plt.plot(train_loss_history, marker="o", label="Training Loss")
plt.plot(val_loss_history, marker="s", label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("plot5.png", dpi=300)
plt.show()


### 16. Evaluation

In [ ]:
# ==========================================
# Task 7 : Model Evaluation on Test Set
# ==========================================

model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

test_accuracy = accuracy_score(all_labels, all_predictions)
test_precision = precision_score(all_labels, all_predictions, average="weighted")
test_recall = recall_score(all_labels, all_predictions, average="weighted")
test_f1 = f1_score(all_labels, all_predictions, average="weighted")

print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)
print(f"Test Accuracy  : {test_accuracy:.4f}")
print(f"Test Precision : {test_precision:.4f}")
print(f"Test Recall    : {test_recall:.4f}")
print(f"Test F1-Score  : {test_f1:.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(all_labels, all_predictions, target_names=class_names))


### 17. Confusion Matrix Plot

In [ ]:
# ==========================================
# Task 7.3 : Confusion Matrix
# ==========================================

conf_matrix = confusion_matrix(all_labels, all_predictions)

fig, ax = plt.subplots(figsize=(9, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=class_names)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True)
plt.title("Confusion Matrix - CIFAR-10 Test Set")
plt.tight_layout()
plt.savefig("plot6.png", dpi=300)
plt.show()


### 18. Results Summary Table

In [ ]:
# ==========================================
# Task 7.4 : Results Summary
# ==========================================

final_train_accuracy = train_acc_history[-1]

results_summary_df = pd.DataFrame({
    "Metric": ["Training Accuracy", "Testing Accuracy", "Precision",
               "Recall", "F1-Score", "Number of Parameters"],
    "Value": [f"{final_train_accuracy:.4f}", f"{test_accuracy:.4f}",
              f"{test_precision:.4f}", f"{test_recall:.4f}",
              f"{test_f1:.4f}", trainable_params]
})

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(results_summary_df.to_string(index=False))


### 19. Final Observations

In [ ]:
# ==========================================
# Final Observations
# ==========================================

print("=" * 60)
print("FINAL OBSERVATIONS")
print("=" * 60)
print(f"1. The CNN achieved a test accuracy of {test_accuracy:.4f}, compared to a")
print(f"   final training accuracy of {final_train_accuracy:.4f}.")
print("2. Increasing kernel size reduces feature map dimensions faster,")
print("   confirming the (N - F + 2P) / S + 1 relationship.")
print("3. Max pooling preserved sharper feature activations than average")
print("   pooling, which produced smoother, blurred outputs.")
print("4. The two-convolution CNN architecture required far fewer parameters")
print("   than an equivalent fully connected network would need.")
